# Capa Silver - Catálogo de Pagos

In [1]:
from spark_session import *
from delta.tables import *

## SparkSession

In [2]:
spark = create_spark_session(
    "retail-dlh-silver",
    configs=[
        configure_delta_extensions,
        configure_delta_catalog,
        configure_s3
    ]
)

## Extracción

In [3]:
# - Obtiene dataframe de la capa bronze
df_payment = spark.table("bronze.payment")

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Carga

In [4]:
if DeltaTable.isDeltaTable(spark, "silver.payments"):
  # - Obtiene tabla delta de la capa silver
  deltaTblSilPayments = DeltaTable.forName(spark, "silver.payments")

  ## --Inserta nuevos regitros, actualiza registros con cambios
  deltaTblSilPayments.alias('SILV') \
    .merge(
      df_payment.alias('BRON'),
      condition='BRON.PAYMENT_ID = SILV.PAYMENT_ID'
    ) \
    .whenMatchedUpdateAll () \
    .whenNotMatchedInsertAll() \
    .execute()
else:
  ## --Inserta registros por primera vez
  df_payment.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver.payments")

## Vacuum

In [5]:
# - Actualiza la configuración para que pueda establecer un período de retención cero.
#spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

#- Ejecutemos el comando de VACUUM 
#spark.sql(f"VACUUM '{vPathSilv}PAYMENTS.delta' RETAIN 0 HOURS").show(truncate=False)

## Cierre de SparkSession

In [6]:
spark.stop()